# Training Dynamics

Read TensorBoard event files from `outputs/<model>__<dataset>__seed<N>/<timestamp>/tensorboard/` and visualize per-run training telemetry:

1. **Loss curves** — train vs. eval loss on the same axes (overfitting / underfitting diagnosis)
2. **Gradient flow** — `train/grad_norm` over steps (vanishing / exploding check)
3. **Learning-rate schedule** — `train/learning_rate` (warmup + decay sanity check)
4. **Cross-model comparison** — same dataset, all models, one metric side-by-side

Use the dropdowns in Section 2 to pick a specific run; the four subplots below refresh together.

In [13]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# OUTPUTS_DIR = Path("Documents/outputs2").resolve()
OUTPUTS_DIR = Path('/Users/basel.alshaikhdeeb/Documents/outputs')
print(f"Scanning: {OUTPUTS_DIR}")
print(f"Exists:   {OUTPUTS_DIR.exists()}")

Scanning: /Users/basel.alshaikhdeeb/Documents/outputs
Exists:   True


## 1 · Discover runs

Walks the `outputs/` tree once and builds a per-run index. Each run entry stores its `(model, dataset, seed, timestamp, tensorboard_dir)`; the actual scalar series are loaded lazily on selection.

In [14]:
def discover_runs(root: Path) -> pd.DataFrame:
    """Return DataFrame with one row per (model, dataset, seed, timestamp) run."""
    records = []
    if not root.exists():
        return pd.DataFrame(records)
    for run_dir in sorted(root.iterdir()):
        if not run_dir.is_dir():
            continue
        name = run_dir.name
        if name.count("__") < 2:
            continue
        model, dataset, seed_tag = name.split("__", 2)
        seed = seed_tag.replace("seed", "")
        for ts_dir in sorted(run_dir.iterdir()):
            if not ts_dir.is_dir():
                continue
            tb_dir = ts_dir / "tensorboard"
            if not tb_dir.exists() or not any(tb_dir.glob("events.out.tfevents.*")):
                continue
            records.append({
                "model": model,
                "dataset": dataset,
                "seed": seed,
                "timestamp": ts_dir.name,
                "tensorboard_dir": str(tb_dir),
            })
    return pd.DataFrame(records)


runs = discover_runs(OUTPUTS_DIR)
print(f"Found {len(runs):,} runs across "
      f"{runs['model'].nunique()} models × "
      f"{runs['dataset'].nunique()} datasets × "
      f"{runs['seed'].nunique()} seeds.")
runs.head(10)

KeyError: 'model'

In [8]:
def load_scalars(tb_dir: str, tags: list[str] | None = None) -> dict[str, pd.DataFrame]:
    """Return {tag: DataFrame(step, wall_time, value)} for a run's TB events.

    ``tags=None`` returns every scalar tag in the log. Reads are cached via
    :func:`functools.lru_cache` on the surrounding wrapper.
    """
    ea = EventAccumulator(tb_dir, size_guidance={"scalars": 0})
    ea.Reload()
    available = ea.Tags().get("scalars", [])
    picks = available if tags is None else [t for t in tags if t in available]
    out: dict[str, pd.DataFrame] = {}
    for tag in picks:
        events = ea.Scalars(tag)
        out[tag] = pd.DataFrame({
            "step": [e.step for e in events],
            "wall_time": [e.wall_time for e in events],
            "value": [e.value for e in events],
        })
    return out


_scalar_cache: dict[str, dict[str, pd.DataFrame]] = {}


def get_scalars(tb_dir: str) -> dict[str, pd.DataFrame]:
    """Cached loader — one hit per tb_dir."""
    if tb_dir not in _scalar_cache:
        _scalar_cache[tb_dir] = load_scalars(tb_dir)
    return _scalar_cache[tb_dir]


if not runs.empty:
    sample_tb = runs.iloc[0]["tensorboard_dir"]
    sample = get_scalars(sample_tb)
    print(f"Sample run tags ({runs.iloc[0]['model']}/{runs.iloc[0]['dataset']}):")
    print("  " + ", ".join(sorted(sample.keys())))

Sample run tags (bert-base/bc5cdr):
  eval/accuracy, eval/loss, eval/macro_f1, eval/macro_precision, eval/macro_recall, eval/micro_f1, eval/micro_precision, eval/micro_recall, eval/roc_auc, eval/runtime, eval/samples_per_second, eval/steps_per_second, eval/weighted_f1, train/epoch, train/grad_norm, train/learning_rate, train/loss, train/total_flos, train/train_loss, train/train_runtime, train/train_samples_per_second, train/train_steps_per_second


## 2 · Per-run inspector

Pick model → dataset → seed. The four subplots refresh together:

* **Top-left:** train/loss vs eval/loss on shared x-axis (steps). Divergence between the two curves marks the point where the model starts overfitting; if both stay flat, it's underfitting.
* **Top-right:** train/grad_norm. Values that trend toward 0 → vanishing gradients; values spiking or plateauing above ~10 → exploding.
* **Bottom-left:** train/learning_rate. Confirms the warmup + cosine/linear decay actually ran.
* **Bottom-right:** eval/micro_f1 and eval/macro_f1 — task quality over the same steps.

In [9]:
def _pick_col(fig_ax, tag: str, scalars: dict, color: str, label: str, log: bool = False):
    df = scalars.get(tag)
    if df is None or df.empty:
        return False
    fig_ax.plot(df["step"], df["value"], label=label, color=color, linewidth=1.4)
    if log:
        fig_ax.set_yscale("log")
    return True


def plot_run(model: str, dataset: str, seed: str) -> None:
    match = runs[(runs["model"] == model) & (runs["dataset"] == dataset) & (runs["seed"] == seed)]
    if match.empty:
        print(f"No run for {model}/{dataset}/seed{seed}")
        return
    # Pick the most recent timestamp if multiple exist.
    row = match.sort_values("timestamp").iloc[-1]
    scalars = get_scalars(row["tensorboard_dir"])

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    fig.suptitle(f"{model}  |  {dataset}  |  seed {seed}  |  {row['timestamp']}", fontsize=12)

    # (1) Loss curves
    ax = axes[0, 0]
    got_train = _pick_col(ax, "train/loss", scalars, "#2b6cb0", "train/loss")
    got_eval = _pick_col(ax, "eval/loss", scalars, "#c53030", "eval/loss")
    ax.set_title("Loss curves (overfitting diagnosis)")
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    if got_train or got_eval:
        ax.legend(loc="upper right")
    ax.grid(alpha=0.3)

    # (2) Gradient norm
    ax = axes[0, 1]
    got = _pick_col(ax, "train/grad_norm", scalars, "#805ad5", "grad_norm", log=True)
    ax.set_title("Gradient flow (train/grad_norm, log scale)")
    ax.set_xlabel("step")
    ax.set_ylabel("grad norm")
    if not got:
        ax.text(0.5, 0.5, "train/grad_norm not logged", ha="center", va="center", transform=ax.transAxes)
    ax.grid(alpha=0.3)

    # (3) Learning rate
    ax = axes[1, 0]
    got = _pick_col(ax, "train/learning_rate", scalars, "#38a169", "lr")
    ax.set_title("Learning-rate schedule")
    ax.set_xlabel("step")
    ax.set_ylabel("lr")
    if not got:
        ax.text(0.5, 0.5, "train/learning_rate not logged", ha="center", va="center", transform=ax.transAxes)
    ax.grid(alpha=0.3)

    # (4) Eval quality
    ax = axes[1, 1]
    plotted = False
    if _pick_col(ax, "eval/micro_f1", scalars, "#d69e2e", "micro F1"):
        plotted = True
    if _pick_col(ax, "eval/macro_f1", scalars, "#2c7a7b", "macro F1"):
        plotted = True
    ax.set_title("Eval quality")
    ax.set_xlabel("step")
    ax.set_ylabel("F1")
    ax.set_ylim(0, 1)
    if plotted:
        ax.legend(loc="lower right")
    else:
        ax.text(0.5, 0.5, "eval F1 not logged", ha="center", va="center", transform=ax.transAxes)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [10]:
# Interactive: model -> dataset -> seed cascade
if runs.empty:
    print("No runs available — populate outputs/ first.")
else:
    model_dd = widgets.Dropdown(options=sorted(runs["model"].unique()), description="model:")
    dataset_dd = widgets.Dropdown(options=[], description="dataset:")
    seed_dd = widgets.Dropdown(options=[], description="seed:")

    def _refresh_datasets(*_):
        sub = runs[runs["model"] == model_dd.value]
        dataset_dd.options = sorted(sub["dataset"].unique())

    def _refresh_seeds(*_):
        sub = runs[(runs["model"] == model_dd.value) & (runs["dataset"] == dataset_dd.value)]
        seed_dd.options = sorted(sub["seed"].unique())

    model_dd.observe(_refresh_datasets, names="value")
    dataset_dd.observe(_refresh_seeds, names="value")
    _refresh_datasets()
    _refresh_seeds()

    ui = widgets.HBox([model_dd, dataset_dd, seed_dd])
    out = widgets.interactive_output(
        plot_run,
        {"model": model_dd, "dataset": dataset_dd, "seed": seed_dd},
    )
    display(ui, out)

Output()

## 3 · Cross-model comparison on one dataset

Fix a dataset + metric and overlay every model's curve on the same axes. Use this to answer questions like "which model's eval loss diverges earliest on DrugProt?" or "whose gradient norm stays healthiest across seeds?"

When multiple seeds exist for a `(model, dataset)` pair, we plot the mean and shade ±1σ across seeds.

In [11]:
def _resample_to_common_steps(dfs: list[pd.DataFrame], n_bins: int = 200) -> pd.DataFrame:
    """Align multiple seed series onto a common step grid by binning + averaging."""
    if not dfs:
        return pd.DataFrame(columns=["step", "mean", "std"])
    min_step = min(df["step"].min() for df in dfs)
    max_step = max(df["step"].max() for df in dfs)
    bins = np.linspace(min_step, max_step, n_bins + 1)
    centers = 0.5 * (bins[1:] + bins[:-1])
    binned = []
    for df in dfs:
        idx = np.digitize(df["step"], bins) - 1
        idx = np.clip(idx, 0, n_bins - 1)
        vals = np.full(n_bins, np.nan)
        sums = np.zeros(n_bins)
        counts = np.zeros(n_bins)
        np.add.at(sums, idx, df["value"].values)
        np.add.at(counts, idx, 1)
        with np.errstate(invalid="ignore", divide="ignore"):
            vals = np.where(counts > 0, sums / counts, np.nan)
        binned.append(vals)
    stacked = np.vstack(binned)
    mean = np.nanmean(stacked, axis=0)
    std = np.nanstd(stacked, axis=0) if stacked.shape[0] > 1 else np.zeros_like(mean)
    return pd.DataFrame({"step": centers, "mean": mean, "std": std}).dropna(subset=["mean"])


def plot_cross_model(dataset: str, tag: str, log_y: bool = False) -> None:
    sub = runs[runs["dataset"] == dataset]
    if sub.empty:
        print(f"No runs for {dataset}")
        return
    fig, ax = plt.subplots(figsize=(11, 5))
    palette = plt.cm.tab20.colors
    per_model = defaultdict(list)
    for _, row in sub.iterrows():
        scalars = get_scalars(row["tensorboard_dir"])
        if tag not in scalars or scalars[tag].empty:
            continue
        per_model[row["model"]].append(scalars[tag])

    for i, (model, dfs) in enumerate(sorted(per_model.items())):
        color = palette[i % len(palette)]
        if len(dfs) == 1:
            df = dfs[0]
            ax.plot(df["step"], df["value"], color=color, label=f"{model} (n=1)", linewidth=1.3)
        else:
            agg = _resample_to_common_steps(dfs)
            if agg.empty:
                continue
            ax.plot(agg["step"], agg["mean"], color=color, label=f"{model} (n={len(dfs)})", linewidth=1.5)
            ax.fill_between(agg["step"], agg["mean"] - agg["std"], agg["mean"] + agg["std"], color=color, alpha=0.15)

    ax.set_title(f"{dataset} — {tag} (mean ± 1σ across seeds)")
    ax.set_xlabel("step")
    ax.set_ylabel(tag)
    if log_y:
        ax.set_yscale("log")
    if per_model:
        ax.legend(loc="best", fontsize=8, ncol=2)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [12]:
if not runs.empty:
    dataset_cmp_dd = widgets.Dropdown(options=sorted(runs["dataset"].unique()), description="dataset:")
    tag_cmp_dd = widgets.Dropdown(
        options=[
            ("train/loss", "train/loss"),
            ("eval/loss", "eval/loss"),
            ("train/grad_norm (log)", "train/grad_norm"),
            ("train/learning_rate", "train/learning_rate"),
            ("eval/micro_f1", "eval/micro_f1"),
            ("eval/macro_f1", "eval/macro_f1"),
        ],
        description="metric:",
    )
    log_cb = widgets.Checkbox(value=False, description="log y-axis")

    def _sync_log(*_):
        log_cb.value = tag_cmp_dd.value == "train/grad_norm"

    tag_cmp_dd.observe(_sync_log, names="value")
    _sync_log()

    ui2 = widgets.HBox([dataset_cmp_dd, tag_cmp_dd, log_cb])
    out2 = widgets.interactive_output(
        plot_cross_model,
        {"dataset": dataset_cmp_dd, "tag": tag_cmp_dd, "log_y": log_cb},
    )
    display(ui2, out2)

Output()

## 4 · How to read the plots

**Loss curves (train vs eval).** Healthy training: both curves decrease, eval decrease lagging slightly. Classic overfitting: train keeps falling while eval flattens then rises. Underfitting: both stay high and roughly flat — usually a capacity or learning-rate issue. Collapse (e.g. BioLinkBERT-large × DrugProt): train_loss stuck near `log(num_classes)` from step 1 — signals fp16 loss-scale collapse or lr too high.

**Gradient flow.** The absolute magnitude matters less than the trajectory. Look for norms falling monotonically toward 0 across many steps (vanishing) or spikes above ~10 that don't get clipped (exploding). `max_grad_norm=1.0` in the trainer means anything above 1 was clipped — you'll see the norm hitting a ceiling of 1.0 during unstable phases.

**Learning-rate schedule.** With `warmup_ratio=0.1` and linear decay you should see a triangular ramp: 0 → peak (at 10% of steps) → 0. If the schedule is flat, warmup didn't fire — check `--extra-override training.warmup_ratio=…`.

**Cross-model comparison.** Overlaying the same metric across models is the fastest way to spot outlier training dynamics — one model whose loss curve looks nothing like the others usually points to an architecture / hyperparameter mismatch rather than a data issue.